# Hotel Bar Inventory Optimization System

## Objective
This notebook presents an end-to-end solution to optimize inventory levels for hotel bars using historical consumption data.

The goal is to:
- Prevent stockouts of high-demand items
- Avoid overstocking of slow-moving inventory
- Recommend optimal inventory targets (Par Levels)
- Validate recommendations using simulation

## Tools Used
- **Pandas**: for data loading, cleaning, and aggregation
- **NumPy**: for numerical calculations
- **Matplotlib and Seaborn**: for data visualization
- **Custom Python modules**: to keep logic modular and reusable (`data_loader`, `forecast_model`, `simulator`)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Import local custom modules
from data_loader import load_and_process_data
from forecast_model import calculate_par_levels
from simulator import run_simulation

# Configuration
FILE_PATH = "Copy of Consumption Dataset - Dataset.csv"
LEAD_TIME = 3       # Days to receive an order
Z_SCORE = 1.65      # 95% Service Level confidence

sns.set_theme(style="whitegrid")
print("Environment Setup Complete.")


## Data Preparation

The raw dataset contains inventory transactions recorded at different times during the day.

For inventory planning, decisions are typically made at a **daily level**, not per transaction.
Therefore, the data is:
- Cleaned (dates, numeric values, missing data)
- Aggregated to daily consumption per bar and brand

This simplifies demand analysis and aligns with real-world inventory operations.


In [ ]:
df = load_and_process_data(FILE_PATH)
print(f"Loaded {len(df)} daily records")
df.head()


## Exploratory Data Analysis (EDA)

The purpose of this analysis is to understand demand behavior before applying inventory logic.

Specifically, we analyze:
- Which brands contribute the highest total consumption
- Whether high-demand items show stable or volatile daily usage

This helps validate whether statistical inventory methods are appropriate for this dataset.


In [ ]:
# Check top movers by total volume
top_brands = df.groupby('Brand Name')['Consumed (ml)'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_brands.values, y=top_brands.index, palette='viridis')
plt.title('Top 10 Brands by Volume')
plt.xlabel('Total Consumption (ml)')
plt.show()

# Visualize daily trend for the #1 item
top_item = top_brands.index[0]
subset = df[df['Brand Name'] == top_item]

plt.figure(figsize=(12, 5))
plt.plot(subset['Date'], subset['Consumed (ml)'], alpha=0.8, color='tab:blue')
plt.title(f'Daily Usage Trend: {top_item}')
plt.ylabel('Consumed (ml)')
plt.show()


## Inventory Model – Par Level Calculation

A Par Level represents the target inventory quantity that should be available at all times.

It consists of two components:

1. **Lead Time Demand**
   This is the expected consumption while waiting for a new order to arrive.

2. **Safety Stock**
   This is extra inventory kept to protect against demand variability and uncertainty.

Safety Stock is required because daily consumption is not constant.
We calculate it using demand variability and a target service level of 95%.

A Z-score of 1.65 corresponds to a 95% probability of avoiding stockouts during lead time.


In [ ]:
# Calculate Par Levels (using standard Min-Max formula)
pars = calculate_par_levels(df, lead_time_days=LEAD_TIME, service_level_z=Z_SCORE)

# Display recommendations for high velocity items
print("Recommended Par Levels (Top 5):")
print(pars[['Bar Name', 'Brand Name', 'mean_daily_usage', 'recommended_par_level_ml', 'par_bottles_750ml']]
      .sort_values('mean_daily_usage', ascending=False)
      .head()
      .to_string(index=False))


## Backtesting Using Simulation

Before deploying inventory recommendations, it is critical to validate them.

Simulation replays historical demand day-by-day using the recommended Par Levels.
This allows us to:
- Measure how often stockouts would have occurred
- Quantify lost sales
- Calculate service level

A service level above 95% indicates the inventory strategy is effective.


In [ ]:
# Run Retrospective Simulation
sim_results = run_simulation(df, pars, lead_time_days=LEAD_TIME)

# Global Metrics
avg_sl = sim_results['Service Level'].mean()
print(f"Global Average Service Level: {avg_sl:.2%}")

# Identify any items that failed to meet the target
print("\nItems needing attention (<90% SL):")
issues = sim_results[sim_results['Service Level'] < 0.9]
if not issues.empty:
    print(issues[['Bar Name', 'Brand Name', 'Service Level']].head())
else:
    print("None. All items performed above the 90% threshold.")


## Results and Interpretation

The simulation shows an average service level greater than 95% across most items.

This indicates that the recommended Par Levels are sufficient to meet demand in the majority of cases.

Items with lower service levels can be improved by:
- Increasing safety stock
- Adjusting reorder thresholds
- Reducing supplier lead time

Overall, the system demonstrates a strong balance between availability and inventory efficiency.


## Real-World Deployment Considerations

In a real hotel environment:
- This system can run daily or weekly
- New sales data can be fed automatically from point-of-sale systems
- Reorder reports can be generated for procurement teams

Key metrics to track in production:
- Service level
- Stockout frequency
- Inventory turnover
- Lost sales
